In [ ]:
import os
import cv2
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

plt.style.use("ggplot")
%matplotlib inline

from tqdm import tqdm_notebook, tnrange
from itertools import chain
from skimage.io import imread, imshow, concatenate_images
from skimage.transform import resize
from skimage.morphology import label
from sklearn.model_selection import train_test_split

import tensorflow as tf

from keras.models import Model, load_model
from keras.layers import Input, BatchNormalization, Activation, Dense, Dropout
from keras.layers import Lambda, RepeatVector, Reshape
from keras.layers import Conv2D, Conv2DTranspose
from keras.layers import MaxPooling2D, GlobalMaxPool2D
from keras.layers import concatenate, add
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [ ]:
from keras.preprocessing.image import array_to_img, img_to_array, load_img
from tensorflow.keras.utils import Sequence
import math

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import random
from tqdm import tqdm

In [ ]:
# Define the directory containing your images
directory = "./data/shriya/SHMOLLI-output-unet-myocardium-update2/SAM_masks"

# Get list of filenames in the directory
filenames = os.listdir(directory)

filename = random.choice(filenames)

# Display each image
#for filename in filenames:
# Construct full path to the image
image_path = os.path.join(directory, filename)

# Load the image
image = Image.open(image_path)

# Display the image
plt.imshow(np.array(image))
plt.colorbar()
plt.title(filename)  # Optional: Set title as filename
plt.axis('off')      # Optional: Turn off axis
plt.show()

In [ ]:
im_width = 256
im_height = 256

In [ ]:
image_dir = "./data/ekchen/Original-images"
mask_dir = "./data/ekchen/image-masks"

batch_size = 8
epochs = 100
image_size = (im_width, im_height)
n_channels = 1

In [ ]:
model_save_file = './data/shriya/myocardium-unet-ethan.h5'

### Data Generator

In [ ]:
class DataGenerator(Sequence):
    def __init__(self, image_dir, mask_dir, split='train', batch_size=32, image_size=(256, 256), n_channels=1, shuffle=True):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.split = split
        self.batch_size = batch_size
        self.image_size = image_size
        self.n_channels = n_channels
        self.shuffle = shuffle
        self.image_filenames = sorted(os.listdir(image_dir))
        self.mask_filenames = sorted(os.listdir(mask_dir))
        self.on_epoch_end()

        X_train, X_test, y_train, y_test = train_test_split(self.image_filenames, self.mask_filenames, test_size=0.2, random_state=42)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

        if self.split == 'train':
            self.image_filenames = X_train
            self.mask_filenames = y_train
        elif self.split == 'valid':
            self.image_filenames = X_valid
            self.mask_filenames = y_valid
        elif self.split == 'test':
            self.image_filenames = X_test
            self.mask_filenames = y_test
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_filenames) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        image_filenames_temp = [self.image_filenames[k] for k in indexes]
        mask_filenames_temp = [self.mask_filenames[k] for k in indexes]
        X, y = self.__data_generation(image_filenames_temp, mask_filenames_temp)
        return X, y

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.image_filenames))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def preprocess_image(self, image):
        # Recolor image to grayscale
        recolored_image = image

        # Normalize the pixel value array to the range [0, 255]
        normalized_array = (recolored_image - np.min(recolored_image)) / (np.max(recolored_image) - np.min(recolored_image)) * 255
        normalized_array = normalized_array.astype(np.uint8)

        # Resize the image to a common size
        resized_image = resize(normalized_array, self.image_size, mode='constant', preserve_range=True)

        return resized_image

    def preprocess_mask(self, mask):
        # Normalize the mask array to the range [0, 1]
        mask[mask == 255] = 1

        # Resize the mask to a common size
        resized_mask = resize(mask, self.image_size, mode='constant', preserve_range=True)

        return resized_mask

    def __data_generation(self, image_filenames_temp, mask_filenames_temp):
        X = np.empty((self.batch_size, *self.image_size, self.n_channels))
        y = np.empty((self.batch_size, *self.image_size, 1))

        for i, (image_filename, mask_filename) in enumerate(zip(image_filenames_temp, mask_filenames_temp)):
            image = Image.open(os.path.join(self.image_dir, image_filename)).convert('L')
            mask = Image.open(os.path.join(self.mask_dir, mask_filename)).convert('L')

            image = np.array(image)
            mask = np.array(mask)

            image = self.preprocess_image(image)
            mask = self.preprocess_mask(mask)

            X[i,] = image[..., np.newaxis]
            y[i,] = mask[..., np.newaxis]  # Ensure mask has the correct shape

        return X, y

In [ ]:
def visualize_image_and_mask(generator, index):
    # Get a batch of data
    X, y = generator[index]

    # Select the first image and mask in the batch
    image = X[0]
    mask = y[0]

    # Display the image
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.title('Image')
    plt.imshow(image.astype(np.uint8))
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title('Mask')
    plt.imshow(mask.squeeze(), cmap='gray')
    plt.axis('off')

    plt.show()

In [ ]:
training_generator = DataGenerator(image_dir, mask_dir, split='train', batch_size=batch_size, image_size=image_size, n_channels=n_channels)

# Visualize the first batch
visualize_image_and_mask(training_generator, index=0)

## Model

In [ ]:
def conv2d_block(input_tensor, n_filters, kernel_size = 3, batchnorm = True):
    """Function to add 2 convolutional layers with the parameters passed to it"""
    # first layer
    x = Conv2D(filters = n_filters, kernel_size = (kernel_size, kernel_size),\
              kernel_initializer = 'he_normal', padding = 'same')(input_tensor)
    if batchnorm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # second layer
    x = Conv2D(filters = n_filters, kernel_size = (kernel_size, kernel_size),\
              kernel_initializer = 'he_normal', padding = 'same')(input_tensor)
    if batchnorm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)

    return x

In [ ]:
def get_unet(input_img, n_filters = 16, dropout = 0.1, batchnorm = True):
    """Function to define the UNET Model"""
    # Contracting Path
    c1 = conv2d_block(input_img, n_filters * 1, kernel_size = 3, batchnorm = batchnorm)
    p1 = MaxPooling2D((2, 2))(c1)
    p1 = Dropout(dropout)(p1)

    c2 = conv2d_block(p1, n_filters * 2, kernel_size = 3, batchnorm = batchnorm)
    p2 = MaxPooling2D((2, 2))(c2)
    p2 = Dropout(dropout)(p2)

    c3 = conv2d_block(p2, n_filters * 4, kernel_size = 3, batchnorm = batchnorm)
    p3 = MaxPooling2D((2, 2))(c3)
    p3 = Dropout(dropout)(p3)

    c4 = conv2d_block(p3, n_filters * 8, kernel_size = 3, batchnorm = batchnorm)
    p4 = MaxPooling2D((2, 2))(c4)
    p4 = Dropout(dropout)(p4)

    c5 = conv2d_block(p4, n_filters = n_filters * 16, kernel_size = 3, batchnorm = batchnorm)

    # Expansive Path
    u6 = Conv2DTranspose(n_filters * 8, (3, 3), strides = (2, 2), padding = 'same')(c5)
    u6 = concatenate([u6, c4])
    u6 = Dropout(dropout)(u6)
    c6 = conv2d_block(u6, n_filters * 8, kernel_size = 3, batchnorm = batchnorm)

    u7 = Conv2DTranspose(n_filters * 4, (3, 3), strides = (2, 2), padding = 'same')(c6)
    u7 = concatenate([u7, c3])
    u7 = Dropout(dropout)(u7)
    c7 = conv2d_block(u7, n_filters * 4, kernel_size = 3, batchnorm = batchnorm)

    u8 = Conv2DTranspose(n_filters * 2, (3, 3), strides = (2, 2), padding = 'same')(c7)
    u8 = concatenate([u8, c2])
    u8 = Dropout(dropout)(u8)
    c8 = conv2d_block(u8, n_filters * 2, kernel_size = 3, batchnorm = batchnorm)

    u9 = Conv2DTranspose(n_filters * 1, (3, 3), strides = (2, 2), padding = 'same')(c8)
    u9 = concatenate([u9, c1])
    u9 = Dropout(dropout)(u9)
    c9 = conv2d_block(u9, n_filters * 1, kernel_size = 3, batchnorm = batchnorm)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c9)
    model = Model(inputs=[input_img], outputs=[outputs])
    return model

In [ ]:
input_img = Input((im_height, im_width, 1), name='img')
model = get_unet(input_img, n_filters=16, dropout=0.05, batchnorm=True)
model.compile(optimizer=Adam(), loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=10, verbose=1),
    ReduceLROnPlateau(factor=0.1, patience=5, min_lr=0.00001, verbose=1),
    ModelCheckpoint(model_save_file, verbose=1, save_best_only=True, save_weights_only=True)
]

In [ ]:
train_generator = DataGenerator(image_dir, mask_dir, split='train', batch_size=batch_size, image_size=(im_height, im_width))
valid_generator = DataGenerator(image_dir, mask_dir, split='valid', batch_size=batch_size, image_size=(im_height, im_width))

## Training

In [ ]:
results = model.fit(train_generator,
                    epochs=100,
                    callbacks=callbacks,
                    validation_data=valid_generator,
                    validation_steps=len(valid_generator))

In [ ]:
plt.figure(figsize=(8, 8))
plt.title("Learning curve")
plt.plot(results.history["loss"], label="loss")
plt.plot(results.history["val_loss"], label="val_loss")
plt.plot( np.argmin(results.history["val_loss"]), np.min(results.history["val_loss"]), marker="x", color="r", label="best model")
plt.xlabel("Epochs")
plt.ylabel("log_loss")
plt.legend();

## Model Evaluation

In [ ]:
model.load_weights(model_save_file)

In [ ]:
# Assuming you have defined a DataGenerator for testing
test_generator = DataGenerator(image_dir, mask_dir, split='test', batch_size=batch_size, image_size=(im_height, im_width))

# Evaluate the model on the testing data
evaluation = model.evaluate_generator(test_generator, steps=len(test_generator))

# Print the evaluation results (e.g., loss and any other metrics)
print("Testing Loss:", evaluation[0])
print("Testing Metrics:", evaluation[1:])

In [ ]:
# Set a random seed for reproducibility
random.seed(42)

# Choose a random index within the range of the generator's length
index = random.randint(0, len(test_generator) - 1)

# Get a batch of images and masks using the index
X_batch, y_batch = test_generator[index]

predictions = model.predict(X_batch)

In [ ]:
def visualize_image_in_batch(X_batch, predictions):
    # Choose a random image index within the batch
    image_index = random.randint(0, batch_size - 1)

    # Plot the original image
    plt.figure(figsize=(10, 6))
    plt.subplot(1, 3, 1)
    plt.imshow(X_batch[image_index].squeeze(), cmap='gray')
    plt.grid(False)
    plt.title('Original Image')

    # Plot the ground truth mask
    plt.subplot(1, 3, 2)
    plt.imshow(y_batch[image_index].squeeze(), cmap='gray')
    plt.grid(False)
    plt.title('Ground Truth Mask')

    # Plot the model's predicted mask
    prediction = (predictions[image_index].squeeze() > .20).astype(np.uint8)

    plt.subplot(1, 3, 3)
    plt.imshow(prediction, cmap='gray')
    plt.grid(False)
    plt.title('Predicted Mask')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_image_in_batch(X_batch, predictions)
visualize_image_in_batch(X_batch, predictions)
visualize_image_in_batch(X_batch, predictions)

In [ ]:
def predict(image, threshold):

    prediction = model.predict(image)

    print(np.min(prediction))

    prediction = np.array(prediction[0])
    prediction = (prediction > threshold).astype(np.uint8)

    return prediction

def preprocess_image(image):
    # Recolor image to grayscale
    recolored_image = image

    # Normalize the pixel value array to the range [0, 255]
    normalized_array = (recolored_image - np.min(recolored_image)) / (np.max(recolored_image) - np.min(recolored_image)) * 255
    normalized_array = normalized_array.astype(np.uint8)

    # Resize the image to a common size
    resized_image = resize(normalized_array, (256,256), mode='constant', preserve_range=True)

    return resized_image

def postprocess_prediction(prediction, area_threshold=0.5):
    contours, _ = cv2.findContours(prediction, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Check if there are any contours found
    if not contours:
        return prediction

    # Find the largest contour based on area
    largest_contour = max(contours, key=cv2.contourArea)

    # Create a mask of the largest contour
    mask = np.zeros_like(prediction)
    cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # Apply the mask to the original image to keep only the largest contour
    result = cv2.bitwise_and(prediction, mask)
    result = result[..., np.newaxis]

    return result


In [ ]:
def check_completeness(contour):
    """Check if myocardial ring has large angular gaps"""
    if len(contour) < 10:
        return 0.0
    
    # Find center and calculate angles to all points
    M = cv2.moments(contour)
    if M["m00"] == 0:
        return 0.0
    
    center = np.array([M["m10"] / M["m00"], M["m01"] / M["m00"]])
    points = contour[:, 0, :]
    
    # Get angles from center to each point
    angles = [np.arctan2(p[1] - center[1], p[0] - center[0]) for p in points]
    angles = sorted([(a + 2*np.pi) % (2*np.pi) for a in angles])
    
    # Find largest angular gap
    gaps = [angles[i+1] - angles[i] for i in range(len(angles)-1)]
    gaps.append((2*np.pi - angles[-1]) + angles[0])  # wrap-around gap
    max_gap = max(gaps)
    
    # Complete rings shouldn't have gaps > 60 degrees
    return max(0, 1 - max(max_gap - np.pi/3, 0) / np.pi)

def check_quality(image):
    
    # Apply binary thresholding to get a binary image
    binary_image = (image > 0).astype(np.uint8)
    
    # Find contours in the binary image
    contours, hierarchy = cv2.findContours(binary_image, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    
    # Initialize a flag to check for donut shape
    contains_donut = False

    #print(contours)
    
    # Create a copy of the original image to draw contours on
    image_with_contours = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    # Count to store the detected circles
    circle_count = 0
    
    # Loop through contours to check for circles
    for i, contour in enumerate(contours):
        if cv2.arcLength(contour, True) > 0:
            circle_count += 1
            
        cv2.drawContours(image_with_contours, [contour], -1, (0, 255, 0), 2)
    
    # Check if there are exactly two circles
    quality = (circle_count == 2)
    
    if (quality == False and contours):
        main_contour = max(contours, key=cv2.contourArea)
        completeness_score = check_completeness(main_contour)
        
        quality = (completeness_score > 0.98)
    
    return quality

In [ ]:
import tifffile

original_training_path = './data/shriya/filtered-test-ShMOLLI-pngimages'
original_masks_path = './data/shriya/filtered-test-ShMOLLI-mask'

image_files = sorted([f for f in os.listdir(original_training_path)])

mask_files = sorted([f for f in os.listdir(original_masks_path) if ".tif" in f])
mask_paths = [os.path.join(original_masks_path, fname) for fname in mask_files if ".tif" in fname]

ground_means = []
pred_means = []

good_predictions_idx = []
    
for i in tqdm(range(200)):
    filename = image_files[i]
    patient_ID = os.path.splitext(os.path.basename(filename))[0]

    #Transform into Image type
    data = Image.open(os.path.join(original_training_path, filename)).convert('L')
    image = preprocess_image(np.array(data))
    image = image[..., np.newaxis]
    image = np.expand_dims(image, axis=0)

    #Get predicted mask
    prediction = predict(image, 0.9)

    prediction = postprocess_prediction(prediction)

    mask_raw = tifffile.imread(mask_paths[i]) 
    mask_resized = resize(mask_raw, (256,256, 1), mode='constant', preserve_range=True)
    mask_resized[mask_resized < 254] = 0
    mask_resized[mask_resized > 254] = 1
    
    ground_truth = mask_resized* image[0] * 4000/255
    prediction_overlay = prediction* image[0] * 4000/255
    
    prediction_overlay_uint8 = np.clip(prediction_overlay * 255 / 4000, 0, 255).astype(np.uint8)

    if check_quality(prediction_overlay_uint8):  # Pass uint8 version
        good_predictions_idx.append(i)

    ground_truth_values = ground_truth[ground_truth > 0]
    prediction_overlay_values = prediction_overlay[prediction_overlay > 0]
    
    ground_means.append(np.mean(ground_truth_values))
    pred_means.append(np.mean(prediction_overlay_values))


In [ ]:
# Create a copy or new lists to store filtered values
filtered_ground_means = []
filtered_pred_means = []

rejected = []

# Filter out values where ground_means > 2000
for i in range(200):
    if ground_means[i] <= 1300:  # Keep only values that are <= 2000
        filtered_ground_means.append(ground_means[i])
        filtered_pred_means.append(pred_means[i])
    else:
        rejected.append(i)
        
mean_filtered_indices = [i for i in range(len(ground_means)) if ground_means[i] <= 1300]
mean_rejected_indices = [i for i in range(len(ground_means)) if ground_means[i] > 1300]


In [ ]:
filtered_indices = list(set(good_predictions_idx) & set(mean_filtered_indices))

filtered_ground_means = [ground_means[i] for i in filtered_indices]
filtered_pred_means = [pred_means[i] for i in filtered_indices]

In [ ]:
from scipy import stats

# Perform paired t-test
t_stat, p_value = stats.ttest_rel(filtered_ground_means, filtered_pred_means)

# Calculate differences and means for Bland-Altman plot
differences = np.array(filtered_pred_means) - np.array(filtered_ground_means)
means = (np.array(filtered_pred_means) + np.array(filtered_ground_means)) / 2

# Calculate mean difference and limits of agreement (1.96 * std)
mean_diff = np.mean(differences)
std_diff = np.std(differences)
lower_limit = mean_diff - 1.96 * std_diff
upper_limit = mean_diff + 1.96 * std_diff

# Create Bland-Altman plot
plt.figure(figsize=(10, 7))
plt.scatter(means, differences, alpha=0.7)
plt.axhline(mean_diff, color='red', linestyle='-', label=f'Mean Difference: {mean_diff:.2f}')
plt.axhline(lower_limit, color='gray', linestyle='--', label=f'Lower Limit: {lower_limit:.2f}')
plt.axhline(upper_limit, color='gray', linestyle='--', label=f'Upper Limit: {upper_limit:.2f}')

# Add reference line at y=0 (perfect agreement)
plt.axhline(0, color='blue', linestyle=':', alpha=0.5, label='Line of Perfect Agreement')

# Calculate R-squared (coefficient of determination)
correlation_coefficient = np.corrcoef(filtered_ground_means, filtered_pred_means)[0, 1]
r_squared = correlation_coefficient ** 2

# Create plot labels and title
plt.xlabel('Mean of Measurements')
plt.ylabel('Difference (Predicted - Ground Truth)')
plt.title('Bland-Altman Plot of Predicted vs Ground Truth Measurements\n' + 
          f'Paired t-test: t={t_stat:.3f}, p={p_value:.4f}, R²={r_squared:.3f}')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

print(f'Paired t-test: t={t_stat:.3f}, p={p_value:.4f}, R²={r_squared:.3f}')

In [ ]:
def visualize_example(example_array):
    plt.imshow(example_array, cmap='coolwarm')
    plt.show()
    
    pixels = example_array[example_array > 0].flatten()

    plt.figure(figsize=(8, 5))
    plt.hist(pixels, bins=300, alpha=0.7, color='skyblue', edgecolor='black', density=True)
    plt.title("Distribution of Values", fontsize=16)
    plt.xlabel("Value", fontsize=12)
    plt.ylabel("Frequency (Normalized)", fontsize=12)
    plt.grid(alpha=0.3)
    plt.show()

    print("Mean: ", np.mean(pixels))

    return

In [ ]:
from random import choice

print("Chosen Mask")
x = choice(filtered_indices)

mask_raw = tifffile.imread(mask_paths[x])
mask_resized = resize(mask_raw, (256,256, 1), mode='constant', preserve_range=True)
mask_resized[mask_resized < 254] = 0
mask_resized[mask_resized > 254] = 1

ground_truth = mask_resized* image[0] * 4000/255

ground_truth_values = ground_truth[ground_truth > 0]

print("Ground truth mask number: ", x)

visualize_example(ground_truth)


print("Rejected Mask")
x = choice(mean_rejected_indices)

mask_raw = tifffile.imread(mask_paths[x])
mask_resized = resize(mask_raw, (256,256, 1), mode='constant', preserve_range=True)
mask_resized[mask_resized < 254] = 0
mask_resized[mask_resized > 254] = 1

ground_truth = mask_resized* image[0] #* 4000/255

ground_truth_values = ground_truth[ground_truth > 0]

print("Ground truth mask number: ", x)

visualize_example(ground_truth)

In [ ]:
def dice_coefficient(ground_truth, prediction, print_values=False):
    # Threshold the arrays to ensure they contain only 0s and 1s
    ground_truth = (ground_truth > .08).astype(np.uint8)
    prediction = (prediction > .08).astype(np.uint8)

    # Resize the arrays to ensure they contain are the same shape
    ground_truth = ground_truth.squeeze()
    prediction = prediction.squeeze()

    intersection = np.multiply(ground_truth, prediction)

    if (print_values == True):
        ("Intersection:", np.sum(intersection))
        print("Ground Truth:", np.sum(ground_truth))
        print("Prediction:", np.sum(prediction))

    dice = (2 * np.sum(intersection)) / (np.sum(ground_truth) + np.sum(prediction))
    return dice

In [ ]:
# Calculate dice scores for filtered images
dice_scores = []

for i in tqdm(filtered_indices):
    filename = image_files[i]
    
    # Load and preprocess image
    data = Image.open(os.path.join(original_training_path, filename)).convert('L')
    image = preprocess_image(np.array(data))
    image = image[..., np.newaxis]
    image = np.expand_dims(image, axis=0)
    
    # Get predicted mask
    prediction = predict(image, 0.9)
    prediction = postprocess_prediction(prediction)
    
    # Load ground truth mask
    mask_raw = tifffile.imread(mask_paths[i])
    mask_resized = resize(mask_raw, (256, 256, 1), mode='constant', preserve_range=True)
    mask_resized[mask_resized < 254] = 0
    mask_resized[mask_resized > 254] = 1
    
    # Calculate dice coefficient
    dice = dice_coefficient(mask_resized, prediction)
    dice_scores.append(dice)

# Print statistics
print(f"\nDice Score Statistics (for ground_means <= 1300):")
print(f"Mean: {np.mean(dice_scores):.4f}")
print(f"Median: {np.median(dice_scores):.4f}")
print(f"Std: {np.std(dice_scores):.4f}")
print(f"Min: {np.min(dice_scores):.4f}")
print(f"Max: {np.max(dice_scores):.4f}")


In [ ]:
def visualize_segmentation_pipeline(image_index, show_ground_truth=True, show_t1_overlay=True):
    """
    Visualize the complete segmentation pipeline for your data:
    1. Original input image
    2. Raw ML prediction
    3. Post-processed result
    4. Optional: Ground truth comparison
    5. Optional: T1 map overlay
    
    Parameters:
    - image_index: Index of the image to visualize (0-199)
    - show_ground_truth: Whether to show ground truth mask
    - show_t1_overlay: Whether to show T1 map overlay
    """
    # Load and preprocess image
    filename = image_files[image_index]
    patient_ID = os.path.splitext(os.path.basename(filename))[0]
    
    data = Image.open(os.path.join(original_training_path, filename)).convert('L')
    image_preprocessed = preprocess_image(np.array(data))
    image_input = image_preprocessed[..., np.newaxis]
    image_input = np.expand_dims(image_input, axis=0)
    
    # Get prediction
    raw_prediction = predict(image_input, 0.9)
    post_processed_mask = postprocess_prediction(raw_prediction)
    
    # Load ground truth mask
    mask_raw = tifffile.imread(mask_paths[image_index])
    mask_resized = resize(mask_raw, (256, 256, 1), mode='constant', preserve_range=True)
    mask_resized[mask_resized < 254] = 0
    mask_resized[mask_resized > 254] = 1
    
    # Create overlays for T1 mapping
    ground_truth_overlay = mask_resized * image_input[0] * 4000/255
    prediction_overlay = post_processed_mask * image_input[0] * 4000/255
    
    # Calculate dice score
    dice = dice_coefficient(mask_resized, post_processed_mask)
    
    # Debug prints
    print(f"Patient ID: {patient_ID}")
    print(f"Original image shape: {image_preprocessed.shape}")
    print(f"Raw prediction shape: {raw_prediction.shape}")
    print(f"Post-processed mask shape: {post_processed_mask.shape}")
    print(f"Ground truth mean: {ground_means[image_index]:.2f}")
    print(f"Prediction mean: {pred_means[image_index]:.2f}")
    print(f"Dice score: {dice:.4f}")
    
    # Determine number of columns
    n_cols = 3
    if show_ground_truth:
        n_cols += 1
    if show_t1_overlay:
        n_cols += 1
    
    plt.figure(figsize=(n_cols*4, 4))
    col = 1
    
    # Plot 1: Input image with white box
    ax1 = plt.subplot(1, n_cols, col)
    plt.imshow(image_preprocessed.squeeze(), cmap='gray')
    plt.title('Input', fontsize=14)
    ax1.grid(False)
    
    # Add white box around heart region
    heart_region_y, heart_region_x = np.where(image_preprocessed.squeeze() > 0.1)
    if len(heart_region_x) > 0 and len(heart_region_y) > 0:
        min_x, max_x = np.min(heart_region_x), np.max(heart_region_x)
        min_y, max_y = np.min(heart_region_y), np.max(heart_region_y)
        width, height = max_x - min_x, max_y - min_y
        rect = plt.Rectangle((min_x, min_y), width, height, 
                           linewidth=2, edgecolor='white', facecolor='none')
        ax1.add_patch(rect)
    col += 1
    
    # Plot 2: ML prediction with yellow contour
    ax2 = plt.subplot(1, n_cols, col)
    plt.imshow(image_preprocessed.squeeze(), cmap='gray')
    contour = ax2.contour(raw_prediction.squeeze() > 0.2, levels=[0.5], 
                         colors=['yellow'], linewidths=2)
    plt.title('Machine\nlearning', fontsize=14)
    ax2.grid(False)
    col += 1
    
    # Plot 3: Post-processed mask with dashed yellow contour
    ax3 = plt.subplot(1, n_cols, col)
    plt.imshow(image_preprocessed.squeeze(), cmap='gray')
    post_mask_binary = post_processed_mask.squeeze() > 0.5
    contour = ax3.contour(post_mask_binary, levels=[0.5], 
                         colors=['yellow'], linestyles='dashed', linewidths=2)
    plt.title('Post-\nprocessing', fontsize=14)
    ax3.grid(False)
    col += 1
    
    # Plot 4: Ground truth (optional)
    if show_ground_truth:
        ax4 = plt.subplot(1, n_cols, col)
        plt.imshow(image_preprocessed.squeeze(), cmap='gray')
        gt_binary = mask_resized.squeeze() > 0.5
        contour = ax4.contour(gt_binary, levels=[0.5], 
                             colors=['cyan'], linewidths=2)
        plt.title(f'Ground Truth\nDice: {dice:.3f}', fontsize=14)
        ax4.grid(False)
        col += 1
    
    # Plot 5: T1 map overlay (optional)
    if show_t1_overlay:
        ax5 = plt.subplot(1, n_cols, col)
        
        # Create T1 map visualization
        t1_display = prediction_overlay.squeeze()
        t1_display_masked = np.ma.masked_where(t1_display == 0, t1_display)
        
        plt.imshow(image_preprocessed.squeeze(), cmap='gray')
        im = plt.imshow(t1_display_masked, cmap='jet', alpha=0.7, vmin=0, vmax=1500)
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax5, fraction=0.046, pad=0.04)
        cbar.set_label('T1 (ms)', rotation=270, labelpad=15)
        
        plt.title(f'T1 Map\nMean: {pred_means[image_index]:.0f} ms', fontsize=14)
        ax5.grid(False)
        col += 1
    
    plt.tight_layout()
    plt.show()


In [ ]:
random_index = choice(filtered_indices)
print("Random Index", random_index)
visualize_segmentation_pipeline(random_index, show_ground_truth=True, show_t1_overlay=True)

random_index = choice(filtered_indices)
print("Random Index", random_index)
visualize_segmentation_pipeline(random_index, show_ground_truth=True, show_t1_overlay=True)

random_index = choice(filtered_indices)
print("Random Index", random_index)
visualize_segmentation_pipeline(random_index, show_ground_truth=True, show_t1_overlay=True)

random_index = choice(filtered_indices)
print("Random Index", random_index)
visualize_segmentation_pipeline(random_index, show_ground_truth=True, show_t1_overlay=True)

random_index = choice(filtered_indices)
print("Random Index", random_index)
visualize_segmentation_pipeline(random_index, show_ground_truth=True, show_t1_overlay=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats

# Set aesthetic parameters for a publication-quality figure
plt.figure(figsize=(8, 7))
sns.set_style("ticks")
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.5

# Calculate correlation coefficient and p-value
correlation, p_value = stats.pearsonr(filtered_ground_means, filtered_pred_means)
r_squared = correlation**2

# Create the scatter plot
ax = plt.subplot(111)
scatter = ax.scatter(filtered_ground_means, filtered_pred_means, alpha=0.7, s=60, 
                    edgecolor='white', linewidth=0.5, color='#3182bd')

# Add line of best fit
x_range = np.linspace(min(filtered_ground_means), max(filtered_ground_means), 100)
slope, intercept, r_value, p_value, std_err = stats.linregress(filtered_ground_means, filtered_pred_means)
plt.plot(x_range, intercept + slope*x_range, '--', color='#e34a33', linewidth=2)

# Add perfect agreement line (y=x)
plt.plot(x_range, x_range, '-', color='#636363', linewidth=1, alpha=0.5)

# Add annotations for correlation statistics
plt.annotate(f'r = {correlation:.3f}\nR² = {r_squared:.3f}\np {("<" if p_value < 0.001 else "=")} {(0.001 if p_value < 0.001 else p_value):.3f}',
             xy=(0.05, 0.95), xycoords='axes fraction',
             bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.8),
             fontsize=11, ha='left', va='top')

# Add axis labels and title
plt.xlabel('Ground Truth Mean T1 Relaxation Time', fontweight='bold')
plt.ylabel('Predicted Mean T1 Relaxation Time', fontweight='bold')
plt.title('Correlation Between Ground Truth and Predicted T1 Time', fontsize=14, pad=20)

# Ensure axes have equal scales
plt.axis('equal')
ax.set_aspect('equal')

# Adjust grid appearance and spines
plt.grid(True, linestyle='--', alpha=0.3)
for spine in ['right', 'top']:
    ax.spines[spine].set_visible(False)

# Set tick marks inward
ax.tick_params(axis='both', direction='in', width=1, length=5)

# Add n value
#plt.annotate(f'n = {len(filtered_ground_means)}', xy=(0.95, 0.05), xycoords='axes fraction',
             fontsize=10, ha='right', va='bottom')

# Tight layout and high-quality output
plt.tight_layout()

print(f'p-value: {p_value}, R2: {r_squared} ')

# Quality Control ID List

In [ ]:
directory
image_files = [os.path.join(directory, filename) for filename in sorted(os.listdir(directory))]

In [ ]:
quality_IDs = []
bad_IDs = []

for file in tqdm(image_files):
    ID = file[91:]
    ID = ID[:7]
    
    image = np.array(Image.open(file))
    image = image[..., np.newaxis]
    
    if check_quality(image):
        quality_IDs.append(ID)
    else:
        bad_IDs.append(ID)

In [ ]:
np.save("./data/shriya/SHMOLLI-output-unet-myocardium-update2/quality_ID_list.csv", quality_IDs)

In [ ]:
np.save("./data/shriya/SHMOLLI-output-unet-myocardium-update2/bad_ID_list.csv", bad_IDs)